# Day 10: Final Analysis & Cross-Shard Query
**Task 1: Build a mechanism to fetch the last 10 messages of a channel when data is scattered.**
**Task 2: Analysis of the entire system.**

In [ ]:
import hashlib
import random
import time

class Message:
    def __init__(self, user_id, channel_id, content):
        self.user_id = user_id
        self.channel_id = channel_id
        self.content = content
        self.timestamp = time.time()  # We need timestamp to sort cross-shard results
        
class Shard:
    def __init__(self, shard_id):
        self.id = shard_id
        self.messages = []
        self.is_active = True

    def store(self, message):
        if self.is_active:
            self.messages.append(message)

class FinalHashShardManager:
    def __init__(self, num_shards):
        self.shards = [Shard(i) for i in range(num_shards)]

    def send_message(self, message):
        message_key = f"{message.user_id}-{message.channel_id}-{id(message)}"
        h = int(hashlib.md5(str(message_key).encode()).hexdigest(), 16)
        shard = self.shards[h % len(self.shards)]
        if shard.is_active:
            shard.store(message)
        else:
            print(f"Failed to write: Shard {shard.id} is down!")
            
    def fetch_last_10(self, channel_id):
        # CROSS-SHARD QUERY LOGIC
        print(f"Gathering messages for channel {channel_id} across all shards...")
        all_channel_messages = []
        shards_checked = 0
        
        for shard in self.shards:
            if not shard.is_active:
                print(f"WARNING: Shard {shard.id} is down. Partial data returned!")
                continue
            
            shards_checked += 1
            # Filter messages belonging to the channel
            shard_msgs = [m for m in shard.messages if m.channel_id == channel_id]
            all_channel_messages.extend(shard_msgs)
            
        print(f"Checked {shards_checked} shards.")
        
        # Sort by timestamp to get the overall timeline correct
        all_channel_messages.sort(key=lambda m: m.timestamp, reverse=True)
        return all_channel_messages[:10]

### Test Cross-Shard Query

In [ ]:
manager = FinalHashShardManager(3)

# Populate test data
for i in range(50):
    manager.send_message(Message(1, 99, f"Msg {i}"))
    time.sleep(0.001)

last_10 = manager.fetch_last_10(99)
print([m.content for m in last_10])

### Test Failure Simulation (One Server Goes Down)

In [ ]:
manager.shards[1].is_active = False # Kill shard 1
print("Trying to send a message...")
manager.send_message(Message(2, 99, "Will this save?")) 
# Notice how we might lose messages mapped to shard 1!

print("Trying to read last 10...")
res = manager.fetch_last_10(99)
print("Returned:", [m.content for m in res])

### Final System Evaluation (Submission Requirements)
    
1. **Which shard failed first and why?**  
   Under User-sharding and Channel-sharding, individual shards failed immediately during sudden spikes (like a viral event or a spambot influencer). We called this a 'hotspot'. One shard was flooded and hit memory limits, while others were practically empty.

2. **Which strategy looked good but failed under spike?**  
   Channel-based sharding looked like the perfect solution initially. It naturally grouped the context of a conversation, making reads extremely fast. However, when tested under a viral spike (many users flocking to the exact same channel), the entire spike was routed to ONE machine, causing it to collapse under load.

3. **What happens if shards increase from 3 → 10?**  
   With the standard `% (num_shards)` hash algorithm, adding new shards changes the denominator. E.g., `Hash(A) % 3` is totally different from `Hash(A) % 10`. This means all previously stored messages are suddenly mapped to the wrong shards! This causes massive cache misses and data unreachability. A real system would need Consistent Hashing to fix this.

4. **What breaks when one shard goes down?**  
   In Hash-based distribution, data is fragmented. If Shard 1 dies, you don't lose an entire channel — you lose *random gaps* in conversations across *all* channels. If we tried to query the history of a channel, the timeline is incomplete and inconsistent.
